# GRPO Multi-GPU Training + vLLM Weight Sync

Single-node **multi-GPU** GRPO training on GSM8K, then **NCCL weight sync** to a running vLLM server.

| | |
|---|---|
| **Model** | `Qwen/Qwen2.5-0.5B-Instruct` |
| **Architecture** | 3 GPUs in one pod — vLLM on GPU 0, GRPO DDP training on GPUs 1 & 2, localhost NCCL sync |
| **Training** | `torchrun --nproc_per_node=2` for distributed data-parallel GRPO |
| **Weights** | `--load-format auto` (real checkpoint, not dummy) |

## Flow

1. Deploy vLLM with real weights (3 GPUs, vLLM uses GPU 0 only)
2. **Baseline** — 10 GSM8K questions via vLLM API, score accuracy
3. **GRPO train** — TRL `GRPOTrainer` via `torchrun` on GPUs 1 & 2 (DDP, 64 samples)
4. **Weight sync** — fresh process loads checkpoint on GPU 1, NCCL broadcast to vLLM on GPU 0
5. **Post-sync eval** — same 10 questions, compare accuracy + Prometheus metrics

## Install dependencies

In [1]:
!python3 -m pip install -q -U kubeflow kubernetes openai requests transformers datasets trl accelerate vllm

In [2]:
import json
import os
import subprocess
import time

import requests
from kubernetes import client as k8s
from openai import OpenAI

## Configure cluster access and vLLM settings

In [3]:
SA_TOKEN_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/token"
SA_NS_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"

if not os.getenv("OPENSHIFT_API_URL") and os.path.exists(SA_TOKEN_PATH):
    os.environ["OPENSHIFT_API_URL"] = "https://kubernetes.default.svc"
    with open(SA_TOKEN_PATH) as f:
        os.environ["NOTEBOOK_USER_TOKEN"] = f.read()
if not os.getenv("NOTEBOOK_NAMESPACE") and os.path.exists(SA_NS_PATH):
    with open(SA_NS_PATH) as f:
        os.environ["NOTEBOOK_NAMESPACE"] = f.read().strip()

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")
NAMESPACE = os.getenv("NOTEBOOK_NAMESPACE", "<your-namespace>")
VLLM_NAME = "grpo-vllm-rollout"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
IMAGE = os.getenv(
    "VLLM_IMAGE",
    "image-registry.openshift-image-registry.svc:5000/grpoxtrainer/vllm-async-init-poc:async-init-poc-20260519-150604",
)
API_KEY = os.getenv("VLLM_API_KEY", "dummy-token")
WEIGHT_TRANSFER_BACKEND = os.getenv("VLLM_WEIGHT_TRANSFER_BACKEND", "nccl")
VLLM_GPU_COUNT = int(os.getenv("VLLM_GPU_COUNT", "3"))
TRAIN_GPU_COUNT = VLLM_GPU_COUNT - 1   # GPUs 1..N for DDP training
VLLM_GPU_MEM_UTIL = os.getenv("VLLM_GPU_MEM_UTIL", "0.4")
VLLM_LOAD_FORMAT = os.getenv("VLLM_LOAD_FORMAT", "auto")
RUN_TRAINING = os.getenv("RUN_GRPO_TRAINING", "true").lower() == "true"

if not api_server or not token:
    raise RuntimeError("OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN must be set.")
if NAMESPACE == "<your-namespace>":
    raise RuntimeError("Set NOTEBOOK_NAMESPACE before running.")

configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = True
for candidate in [
    "/var/run/secrets/kubernetes.io/serviceaccount/ca.crt",
    os.getenv("SSL_CERT_FILE"),
    os.getenv("REQUESTS_CA_BUNDLE"),
]:
    if candidate and os.path.exists(candidate):
        configuration.ssl_ca_cert = candidate
        break
configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = k8s.ApiClient(configuration)
apps = k8s.AppsV1Api(api_client)
core = k8s.CoreV1Api(api_client)

service_host = f"{VLLM_NAME}.{NAMESPACE}.svc.cluster.local"
service_root = f"http://{service_host}:8000"
base_url = f"{service_root}/v1"
health_url = f"{service_root}/health"

print(f"Namespace: {NAMESPACE}")
print(f"vLLM service: {service_host}")
print(f"Image: {IMAGE}")
print(f"Total GPUs: {VLLM_GPU_COUNT} (vLLM: GPU 0, training DDP: GPUs 1-{VLLM_GPU_COUNT-1})")
print(f"Load-format: {VLLM_LOAD_FORMAT}")
print(f"Run GRPO training: {RUN_TRAINING}")

Namespace: grpoxtrainer
vLLM service: grpo-vllm-rollout.grpoxtrainer.svc.cluster.local
Image: image-registry.openshift-image-registry.svc:5000/grpoxtrainer/vllm-async-init-poc:async-init-poc-20260519-150604
Total GPUs: 3 (vLLM: GPU 0, training DDP: GPUs 1-2)
Load-format: auto
Run GRPO training: True


## Deploy vLLM (3 GPUs, real weights, NCCL weight transfer)

In [4]:
cache_dir = "/tmp/hf-cache"
home_dir = "/tmp/vllm-home"
flashinfer_dir = "/tmp/flashinfer-cache"

deployment = {
    "apiVersion": "apps/v1",
    "kind": "Deployment",
    "metadata": {"name": VLLM_NAME, "namespace": NAMESPACE},
    "spec": {
        "replicas": 1,
        "selector": {"matchLabels": {"app": VLLM_NAME}},
        "template": {
            "metadata": {"labels": {"app": VLLM_NAME}},
            "spec": {
                "containers": [
                    {
                        "name": "vllm",
                        "image": IMAGE,
                        "args": [
                            MODEL_ID,
                            "--host", "0.0.0.0",
                            "--port", "8000",
                            "--gpu-memory-utilization", VLLM_GPU_MEM_UTIL,
                            "--max-model-len", "4096",
                            "--enforce-eager",
                            "--load-format", VLLM_LOAD_FORMAT,
                            "--weight-transfer-config",
                            json.dumps({"backend": WEIGHT_TRANSFER_BACKEND}),
                        ],
                        "ports": [{"containerPort": 8000, "name": "http"}],
                        "env": [
                            {"name": "HF_TOKEN", "value": os.getenv("HF_TOKEN", "")},
                            {"name": "HOME", "value": home_dir},
                            {"name": "HF_HOME", "value": cache_dir},
                            {"name": "HUGGINGFACE_HUB_CACHE", "value": cache_dir},
                            {"name": "TRANSFORMERS_CACHE", "value": cache_dir},
                            {"name": "XDG_CACHE_HOME", "value": "/tmp"},
                            {"name": "VLLM_CACHE_ROOT", "value": "/tmp/vllm-cache"},
                            {"name": "VLLM_CONFIG_ROOT", "value": "/tmp/vllm-config"},
                            {"name": "FLASHINFER_WORKSPACE_DIR", "value": flashinfer_dir},
                            {"name": "VLLM_SERVER_DEV_MODE", "value": "1"},
                            {"name": "VLLM_ALLOW_INSECURE_SERIALIZATION", "value": "1"},
                        ],
                        "resources": {
                            "requests": {
                                "cpu": "4",
                                "memory": "24Gi",
                                "nvidia.com/gpu": str(VLLM_GPU_COUNT),
                            },
                            "limits": {
                                "cpu": "4",
                                "memory": "24Gi",
                                "nvidia.com/gpu": str(VLLM_GPU_COUNT),
                            },
                        },
                        "volumeMounts": [
                            {"name": "hf-cache", "mountPath": cache_dir},
                            {"name": "vllm-home", "mountPath": home_dir},
                            {"name": "flashinfer-cache", "mountPath": flashinfer_dir},
                        ],
                    }
                ],
                "volumes": [
                    {"name": "hf-cache", "emptyDir": {}},
                    {"name": "vllm-home", "emptyDir": {}},
                    {"name": "flashinfer-cache", "emptyDir": {}},
                ],
            },
        },
    },
}
service = {
    "apiVersion": "v1",
    "kind": "Service",
    "metadata": {"name": VLLM_NAME, "namespace": NAMESPACE},
    "spec": {
        "selector": {"app": VLLM_NAME},
        "ports": [{"port": 8000, "targetPort": 8000, "name": "http"}],
    },
}

try:
    apps.create_namespaced_deployment(namespace=NAMESPACE, body=deployment)
except Exception:
    apps.patch_namespaced_deployment(name=VLLM_NAME, namespace=NAMESPACE, body=deployment)

try:
    core.create_namespaced_service(namespace=NAMESPACE, body=service)
except Exception:
    core.patch_namespaced_service(name=VLLM_NAME, namespace=NAMESPACE, body=service)

print(f"Created or updated Deployment/Service for {VLLM_NAME}")

Created or updated Deployment/Service for grpo-vllm-rollout


## Wait for vLLM health

In [5]:
print("Waiting for deployment rollout to finish...")
for _ in range(60):
    dep = apps.read_namespaced_deployment(name=VLLM_NAME, namespace=NAMESPACE)
    ready = dep.status.ready_replicas or 0
    updated = dep.status.updated_replicas or 0
    desired = dep.spec.replicas or 1
    if ready >= desired and updated >= desired:
        print(f"  Rollout complete: {ready}/{desired} ready, {updated} updated")
        break
    print(f"  Rollout in progress: {ready}/{desired} ready, {updated} updated")
    time.sleep(10)
else:
    raise RuntimeError("Deployment rollout did not complete in time")

print("Waiting for vLLM /health endpoint...")
for _ in range(90):
    try:
        if requests.get(health_url, timeout=5).status_code == 200:
            print("vLLM health check passed")
            break
    except Exception:
        pass
    time.sleep(10)
else:
    raise RuntimeError("vLLM did not become healthy in time")

Waiting for deployment rollout to finish...
  Rollout complete: 1/1 ready, 1 updated
Waiting for vLLM /health endpoint...
vLLM health check passed


## GRPO multi-GPU training + weight sync (inside vLLM pod)

Copies `scripts/grpo_vllm_train_sync.py` into the pod, then runs two phases:

- **Phase A** — `torchrun --nproc_per_node=2` on GPUs 1 & 2 (`--train-only`): baseline eval, DDP GRPO training, save checkpoint
- **Phase B** — Plain python on GPUs 0 & 1 (`--sync-only`): load checkpoint, NCCL sync to vLLM, post-sync eval + Prometheus

In [6]:
if not RUN_TRAINING:
    print("Skipping GRPO training. Set RUN_GRPO_TRAINING=true to run.")
    result = None
else:
    vllm_pods = core.list_namespaced_pod(
        namespace=NAMESPACE, label_selector=f"app={VLLM_NAME}"
    )
    vllm_pod_name = vllm_pods.items[0].metadata.name
    print(f"Target vLLM pod: {vllm_pod_name}")

    script_src = os.path.join(
        os.path.dirname(os.path.abspath(".")),
        "scripts",
        "grpo_vllm_train_sync.py",
    )
    if not os.path.exists(script_src):
        script_src = "/opt/app-root/src/grpo_vllm_train_sync.py"
    if not os.path.exists(script_src):
        raise FileNotFoundError(
            "grpo_vllm_train_sync.py not found — copy scripts/ to workbench or run from repo root"
        )

    remote_script = "/tmp/grpo_vllm_train_sync.py"
    subprocess.run(
        ["oc", "cp", script_src, f"{NAMESPACE}/{vllm_pod_name}:{remote_script}"],
        check=True,
    )
    print(f"Copied {script_src} -> {vllm_pod_name}:{remote_script}")

    subprocess.run(
        ["oc", "exec", vllm_pod_name, "-n", NAMESPACE, "--",
         "pip", "install", "-q", "trl", "datasets", "accelerate"],
        check=True,
    )

    train_gpus = ",".join(str(i) for i in range(1, VLLM_GPU_COUNT))  # "1,2"
    sync_gpus = "0,1"

    def _stream_exec(cmd_desc, bash_cmd):
        """Run oc exec with streaming output, return combined stdout."""
        print(f"\n--- {cmd_desc} (streaming logs) ---\n")
        proc = subprocess.Popen(
            ["oc", "exec", vllm_pod_name, "-n", NAMESPACE, "--",
             "bash", "-c", bash_cmd],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        lines = []
        for line in proc.stdout:
            print(line, end="", flush=True)
            lines.append(line)
        proc.wait()
        return proc.returncode, "".join(lines)

    # Phase A: multi-GPU GRPO training via torchrun
    rc_a, out_a = _stream_exec(
        f"Phase A: torchrun DDP training on GPUs [{train_gpus}]",
        f"CUDA_VISIBLE_DEVICES={train_gpus} NCCL_DEBUG=WARN "
        f"torchrun --nproc_per_node={TRAIN_GPU_COUNT} "
        f"{remote_script} --train-only",
    )
    if rc_a != 0:
        raise RuntimeError(f"Phase A (training) failed with exit {rc_a}")
    print("\nPhase A complete — checkpoint saved.")

    # Phase B: weight sync from checkpoint to vLLM
    rc_b, out_b = _stream_exec(
        f"Phase B: weight sync (GPU 1 -> vLLM GPU 0)",
        f"CUDA_VISIBLE_DEVICES={sync_gpus} NCCL_DEBUG=WARN "
        f"python3 -u {remote_script} --sync-only",
    )
    if rc_b != 0:
        raise RuntimeError(f"Phase B (sync) failed with exit {rc_b}")

    class _Result:
        pass
    result = _Result()
    result.stdout = out_a + out_b
    result.returncode = 0

    print("\nMulti-GPU GRPO training + weight sync completed successfully.")

Target vLLM pod: grpo-vllm-rollout-76b4c5dd6f-lklzj


Copied /opt/app-root/src/grpo_vllm_train_sync.py -> grpo-vllm-rollout-76b4c5dd6f-lklzj:/tmp/grpo_vllm_train_sync.py



--- Phase A: torchrun DDP training on GPUs [1,2] (streaming logs) ---



W0520 08:01:17.177000 736 torch/distributed/run.py:851] 


W0520 08:01:17.177000 736 torch/distributed/run.py:851] *****************************************


...

  GRPO train time:      176.8s


  Weight sync time:     0.46s


__RESULTS_JSON__{"pre_eval": {"label": "BEFORE_TRAINING", "n": 10, "correct": 1, "accuracy": 0.1, "avg_latency_s": 0.1695, "results": [{"i": 0, "question": "Janet\u2019s ducks lay 16 eggs per day. She eats three for breakfast every morning an", "ref": "18", "pred": "320", "correct": false, "latency_s": 0.663, "snippet": "#### 320"}, {"i": 1, "question": "A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bol", "ref": "3", "pred": "4", "correct": false, "latency_s": 0.086, "snippet": "#### 4"}, {"i": 2, "question": "Josh decides to try flipping a house.  He buys a house for $80,000 and then puts", "ref": "70000", "pred": "25000", "correct": false, "latency_s": 0.147, "snippet": "#### 25000"}, {"i": 3, "question": "James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  ", "ref": "540", "pred": "180", "correct": false, "latency_s": 0.116, "snippet": "#### 180"}, {"i": 4, "question": "Every day, Wendi feeds each of her chickens three cups 

=== GRPO TRAINING + WEIGHT SYNC COMPLETE ===



Multi-GPU GRPO training + weight sync completed successfully.


## Results summary (accuracy + Prometheus)

In [7]:
if RUN_TRAINING and result is not None and result.stdout:
    marker = "__RESULTS_JSON__"
    results_data = None
    for line in result.stdout.splitlines():
        if marker in line:
            results_data = json.loads(line.split(marker, 1)[1])
            break

    if results_data:
        from IPython.display import HTML, display

        pre = results_data["pre_eval"]
        post = results_data["post_eval"]
        pre_m = results_data["pre_metrics"]
        post_m = results_data["post_metrics"]
        train_s = results_data["train_elapsed_s"]
        sync_s = results_data["sync_elapsed_s"]
        r0 = results_data["reward_mean_first"]
        r1 = results_data["reward_mean_last"]
        train_ws = results_data.get("train_world_size", 1)

        acc_delta = post["accuracy"] - pre["accuracy"]
        acc_color = "green" if acc_delta > 0 else ("red" if acc_delta < 0 else "black")

        eval_rows = ""
        for label, ev in [("Before training", pre), ("After training + sync", post)]:
            eval_rows += (
                f"<tr><td>{label}</td>"
                f"<td>{ev['correct']}/{ev['n']}</td>"
                f"<td>{ev['accuracy']:.1%}</td>"
                f"<td>{ev['avg_latency_s']:.3f}s</td></tr>\n"
            )

        prom_rows = ""
        for k in sorted(set(pre_m.keys()) | set(post_m.keys())):
            short = k.replace("vllm:", "")
            bv, av = pre_m.get(k, 0), post_m.get(k, 0)
            prom_rows += (
                f"<tr><td>{short}</td><td>{bv:.4f}</td>"
                f"<td>{av:.4f}</td><td>{av - bv:+.4f}</td></tr>\n"
            )

        detail_rows = ""
        for phase, ev in [("Before", pre), ("After", post)]:
            for row in ev.get("results", [])[:5]:
                mark = "OK" if row["correct"] else "MISS"
                detail_rows += (
                    f"<tr><td>{phase}</td><td>{mark}</td>"
                    f"<td>{row['question']}</td>"
                    f"<td>{row['ref']}</td><td>{row['pred']}</td></tr>\n"
                )

        html = f"""
        <h3>GRPO Multi-GPU Training + vLLM Weight Sync — Results</h3>
        <p><b>Training GPUs:</b> {train_ws} (DDP) &nbsp;|&nbsp;
        <b>GRPO reward/mean:</b> {r0:.4f} &rarr; {r1:.4f} &nbsp;|&nbsp;
        <b>Train:</b> {train_s}s &nbsp;|&nbsp; <b>Weight sync:</b> {sync_s}s</p>
        <h4>GSM8K accuracy (10 test questions via vLLM)</h4>
        <table border="1" cellpadding="6" style="border-collapse:collapse;font-family:monospace;">
        <tr style="background:#f0f0f0"><th>Phase</th><th>Correct</th><th>Accuracy</th><th>Avg latency</th></tr>
        {eval_rows}
        <tr><td colspan="4" style="color:{acc_color};font-weight:bold;">
        Accuracy delta: {acc_delta:+.1%}</td></tr>
        </table>
        <br/>
        <h4>Sample questions (first 5)</h4>
        <table border="1" cellpadding="6" style="border-collapse:collapse;font-family:monospace;font-size:12px;">
        <tr style="background:#f0f0f0"><th>Phase</th><th></th><th>Question</th><th>Ref</th><th>Pred</th></tr>
        {detail_rows}
        </table>
        <br/>
        <h4>Prometheus vLLM metrics</h4>
        <table border="1" cellpadding="6" style="border-collapse:collapse;font-family:monospace;">
        <tr style="background:#f0f0f0"><th>Metric</th><th>Before</th><th>After</th><th>Delta</th></tr>
        {prom_rows}
        </table>
        """
        display(HTML(html))
    else:
        print("Could not parse __RESULTS_JSON__ from script output.")
else:
    print("No training results to display.")